# Datamine STATCOM Process Example

This notebook demonstrates how to configure and run the **`statcom`** process wrapper in `dmstudio`.

## Process Description

## STATCOM

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Compute comparison summary statistics for validating a model versus samples.

Optionally, output data to Excel to display a summary graph representing a comparison of model and sample statistics by domain.

A samples file and a corresponding model file must exist. Both must contain a numeric value for analysis. Typically, this would be a grade value, although any numeric value can be used. The attribute does not have to be of the same name between the two files, and either or both can contain absent or empty values.

Optionally, you can specify a domain field to segregate results per domain. If Excel output is selected (@**EXCEL** =1), comparison results between sample and model data for each domain will be reported and displayed. The designated * **DOMAIN** field should exist in both sample and model files if one is selected. It is not necessary for all domain keyfield values to exist in both sample and model files; they can exist solely in one file or the other, or one can contain a subset of the partner file values.

A decluster weighting field (* **DCWEIGHT**) can be specified. This attribute will be in the input samples file. It is used to weight sample values for comparison with model values per domain.

A density field (* **DENSITY**) can be specified to select the density values in the model. The grade statistics for the model will then weighted by the cell volume*density ie by tonnage. If a density field is not selected then the model grade statistics will be volume weighted.

The density field is also used in the calculation of the total tonnage for each domain. This statistic is reported in the output tables.

A density parameter (@**DENSITY**) is also available. If a density field has been selected but some of the values in the model are absent data then they will be replaced by the density parameter value. If a density field has not been selected then the density parameter will be used in the calculation of the total tonnage for each domain.

### Excel Output

**Note** : Microsoft Excel 2010 or later is required on the local PC in order to view output from this command.

Excel output is controlled by the @**EXCEL** parameter, which is enabled (=1) by default. If set to 0, only &**OUT** is produced.

If enabled, two additional CSV format files are created in the project directory:

* statcom_parameters.csv: a file that is used by the Excel macro to provide summary information.
* statcom_table.csv: a comma-delimited version of &OUT. This is used for chart calculation

Excel will be launched. Security settings for Excel must be such that Excel macros can be launched. You may need to "Enable content" when Excel launches. If macros and automation are prohibited by your Excel Trust Center settings, you will not be able to view automated chart and summary data.

#### @EXCEL=1 Example

The selected samples file contains 5 lithology attributes, indicated by a numeric value in the **NLITH** column ranging from 0-4 inclusive. The corresponding model also contains an **NLITH** column, but only contains records relating to 2 of those domains. Both sample and model files contain absent grade data at various positions in their respective tables.

A weighted field (* **DCWEIGHT**) in the samples file, so has been specified. Absent values also exist for the weighting field, so only samples with a non-absent weighting value will be calculated.

In this example, **NLITH** =0 has already been coded using the [assign-lithology](<../COMMON/Assign_Sample_Lithologies.md>) command. It represents records where grade data is absent. This value does not occur in the model file.

The resulting Excel sheet includes a table on the "STATCOM_1" worksheet that shows the 5 lithology attribute as data rows.

The chart shows 4 clustered bars, not 5:

[![](../Images/STATCOM_2.jpg)](<javascript:void\(0\);>)

* **NLITH** =0 is not displayed on the chart as the number of non-absent records in the sample file (SNSAMPS) and the model file (MNSAMPS) are both zero. As such, no output can be calculated.

* **NLITH** =1 and **NLITH** =4 (far right and left clusters) show only two bars representing the sample mean grade (SMEAN) and the sample weighted mean grade (WMEAN). No Model mean grade (MMEAN) is shown because the model file contains no non-absent grade records for either domain.

* **NLITH** =2 and **NLITH** =3 show the full set of 3 bars; both sample and model grade data for these domains exists, along with weighting information, so summary statistics have been calculated and can be compared.

Where sample, weighted sample and model mean grade data exists, two additional chart layers are applied, as line graphs;

* The % Variance between model and samples
* The % Variance between model and weighted samples

##### Box and Whisker Plots

The output workbook also includes a "Box & Whisker" worksheet. This includes, for each domain (or across the global data set if domain control is not exercised), a five-number summary for unweighted samples, weighted samples and model cells, for example:

[![](../Images/STATCOM_3.png)](<javascript:void\(0\);>)

### Input Files:

* **samples** (Samples file):
  Input sample file.
  Required=Yes

* **model** (Block model):
  Input model file.
  Required=Yes

### Output Files:

* **out** (Table):
  Output file. This will contain the fields:
  Required=No

### Fields:

* **mgrade** (Alphanumeric : MODEL):
  Model field for statistics.
  Default=Undefined
  Required=Yes

* **sgrade** (Alphanumeric : SAMPLES):
  Sample field for statistics.
  Default=Undefined
  Required=Yes

* **domain** (Alphanumeric : MODEL and SAMPLES):
  Domain keyfield for statistics. Typically this would define an estimation domain.
  Default=Undefined
  Required=No

* **dcweight** (Alphanumeric : SAMPLES):
  Weighting field. Field used for weighting the samples. Typically this would be a
  declustered weight field derived from the **[GRIDDC](<griddc.md>)** or
  **[DECLUST](<declust.md>)** processes
  Default=Undefined
  Required=No

* **density** (Alphanumeric : MODEL):
  Density field to enable calculation of tonnage weighted grade statistics for the model.
  If not selected a global density will be defined by the @DENSITY parameter.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('statcom')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute statcom
print("Running statcom...")
dm_cmd.statcom(
    samples_i=['t_assays'],  # required
    model_i='t_mod1',  # required
    out_o='t_statcom_out',  # required
    mgrade_f='optional',  # required
    sgrade_f='optional',  # required
    # domain_f='optional',  # optional
    # dcweight_f='optional',  # optional
    # density_f='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("statcom execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_statcom_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")